# 02 — Chunking Strategy Experiments

Compare fixed-size vs semantic chunking, sweep chunk sizes and overlaps,
and pick the configuration that produces the best retrieval signal.

**Strategies tested:**
- `RecursiveCharacterTextSplitter` (current)
- `CharacterTextSplitter`
- `SemanticChunker` (requires `langchain-experimental`)

In [ ]:
import sys, os
from pathlib import Path

repo_root = str(Path(os.getcwd()).parents[1] / "RAG_Ai_Assistant_Uni")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import pandas as pd
import matplotlib.pyplot as plt
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from app.scraper import scrape_page

SAMPLE_URL = "https://alphawave.hr/"

## 1. Load Sample Text

In [ ]:
page = scrape_page(SAMPLE_URL)
sample_text = page["content"]
print(f"Sample text: {len(sample_text)} chars, {sample_text.count(chr(10))} lines")
print(sample_text[:300])

## 2. Chunk Size Sweep (RecursiveCharacter)

Hold overlap constant at 15% of chunk size. Compare count, avg length, min/max.

In [ ]:
SIZES = [300, 400, 600, 800, 1000, 1200, 1600]

rows = []
for size in SIZES:
    overlap = int(size * 0.15)
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = splitter.split_text(sample_text)
    lengths = [len(c) for c in chunks]
    rows.append({
        "chunk_size": size,
        "overlap": overlap,
        "count": len(chunks),
        "avg_len": round(sum(lengths) / len(lengths), 0),
        "min_len": min(lengths),
        "max_len": max(lengths),
    })

df_sizes = pd.DataFrame(rows)
display(df_sizes)

## 3. Overlap Sweep (chunk_size=800 fixed)

In [ ]:
FIXED_SIZE = 800
OVERLAPS   = [0, 50, 80, 120, 160, 200]

rows = []
for ov in OVERLAPS:
    splitter = RecursiveCharacterTextSplitter(chunk_size=FIXED_SIZE, chunk_overlap=ov)
    chunks = splitter.split_text(sample_text)
    lengths = [len(c) for c in chunks]
    rows.append({
        "overlap": ov,
        "count": len(chunks),
        "avg_len": round(sum(lengths) / len(lengths), 0),
    })

df_overlaps = pd.DataFrame(rows)
display(df_overlaps)

## 4. Strategy Comparison

Same chunk_size=800, overlap=120 across splitter types.

In [ ]:
SIZE, OV = 800, 120

strategies = {
    "RecursiveCharacter": RecursiveCharacterTextSplitter(chunk_size=SIZE, chunk_overlap=OV),
    "Character (\\n\\n)" : CharacterTextSplitter(separator="\n\n", chunk_size=SIZE, chunk_overlap=OV),
    "Character (\\n)"   : CharacterTextSplitter(separator="\n",   chunk_size=SIZE, chunk_overlap=OV),
}

# Semantic chunker requires langchain-experimental + an embedding model
# Uncomment if installed:
# from langchain_experimental.text_splitter import SemanticChunker
# from langchain_ollama import OllamaEmbeddings
# strategies["Semantic"] = SemanticChunker(OllamaEmbeddings(model="nomic-embed-text"))

rows = []
for name, splitter in strategies.items():
    chunks = splitter.split_text(sample_text)
    lengths = [len(c) for c in chunks]
    rows.append({
        "strategy": name,
        "count": len(chunks),
        "avg_len": round(sum(lengths) / len(lengths), 0),
        "min_len": min(lengths),
        "max_len": max(lengths),
        "example": chunks[0][:120].replace("\n", " ") + "...",
    })

df_strategies = pd.DataFrame(rows)
display(df_strategies[["strategy", "count", "avg_len", "min_len", "max_len"]])

## 5. Chunk Length Distribution

In [ ]:
fig, axes = plt.subplots(1, len(strategies), figsize=(5 * len(strategies), 4), sharey=True)

for ax, (name, splitter) in zip(axes, strategies.items()):
    chunks = splitter.split_text(sample_text)
    lengths = [len(c) for c in chunks]
    ax.hist(lengths, bins=20, edgecolor="black")
    ax.set_title(name)
    ax.set_xlabel("chunk length (chars)")
    ax.set_ylabel("count")

plt.suptitle("Chunk Length Distributions", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Chosen Config — Ingest Test

Set `CHOSEN_SIZE` and `CHOSEN_OVERLAP`, then optionally run the full ingest.
Update `app/scraper.py` defaults once you've decided.

In [ ]:
CHOSEN_SIZE    = 800
CHOSEN_OVERLAP = 120

from app.chunking import chunk_text
chunks = chunk_text(sample_text, chunk_size=CHOSEN_SIZE, overlap=CHOSEN_OVERLAP)
print(f"Config: size={CHOSEN_SIZE}, overlap={CHOSEN_OVERLAP}")
print(f"Chunks : {len(chunks)}")
print(f"Sample chunk:\n{chunks[0]}")